# Training of a WGAN on disk images.

In [ ]:
from pathlib import Path
from typing import Any

import equinox as eqx
import jax
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt
import optax
import scienceplots  # noqa: F401
from mpl_toolkits.axes_grid1 import make_axes_locatable

from organic.model_training import train_wgan
from organic.training_data import TrainingImgLoader

# Set matplotlib settings

In [ ]:
# set scienceplots style and matplotlib settings
plt.style.use(["science", "bright", "no-latex"])

new_rcParams = {
    "image.cmap": "inferno",
    "font.family": "serif",
    "figure.dpi": 300,
    "font.size": 8,
    "xtick.direction": "out",
    "ytick.direction": "out",
}
plt.rcParams.update(new_rcParams)

## Defining generator and critic class

In [ ]:
class GeneratorDisksNIROld(eqx.Module):
    """WGAN generator for monochromatic NIR disk images.

    **Attributes**

    - `layers`: Tuple containing the different neural network layers. The only
        requirement is that each layer should just be callable using the output of the
        preceding layer. The first layer needs to be able to accept a latent 'noise'
        vector of size `size_in`.
    - `size_in`: Size of the latent 'noise' input vector to the first layer.
    """

    layers: tuple[Any, ...]  # Different callable layers.
    size_in: int = eqx.field(static=True)  # Size of input latent vector.

    def __init__(self, key: jax.Array) -> None:
        """**Arguments**

        - `key`: JAX PRNG key to initialise the model.
        """
        key, subkey1, subkey2, subkey3, subkey4, subkey5 = jr.split(key, 6)

        self.size_in = 100
        self.layers = (
            eqx.nn.Linear(
                100, 256 * 16 * 16, use_bias=True, key=subkey1
            ),  # Fully connected.
            eqx.nn.Lambda(
                lambda x: jax.nn.leaky_relu(x, negative_slope=0.1)
            ),  # Activation.
            eqx.nn.Lambda(
                lambda x: jnp.reshape(x, shape=(256, 16, 16))
            ),  # Reshape to 256 x 16 x 16.
            eqx.nn.ConvTranspose2d(
                in_channels=256,
                out_channels=128,
                kernel_size=(4, 4),
                stride=(2, 2),
                padding="SAME",
                padding_mode="ZEROS",
                use_bias=True,
                key=subkey2,
            ),  # Transposed convolution to 128 x 32 x 32.
            eqx.nn.Lambda(
                lambda x: jax.nn.leaky_relu(x, negative_slope=0.1)
            ),  # Activation.
            eqx.nn.ConvTranspose2d(
                in_channels=128,
                out_channels=64,
                kernel_size=(4, 4),
                stride=(2, 2),
                padding="SAME",
                padding_mode="ZEROS",
                use_bias=True,
                key=subkey3,
            ),  # Transposed convolution to 64 x 64 x 64.
            eqx.nn.Lambda(
                lambda x: jax.nn.leaky_relu(x, negative_slope=0.1)
            ),  # Activation.
            eqx.nn.ConvTranspose2d(
                in_channels=64,
                out_channels=32,
                kernel_size=(4, 4),
                stride=(2, 2),
                padding="SAME",
                padding_mode="ZEROS",
                use_bias=True,
                key=subkey4,
            ),  # Transposed convolution to 32 x 128 x 128.
            eqx.nn.Lambda(
                lambda x: jax.nn.leaky_relu(x, negative_slope=0.1)
            ),  # Activation.
            eqx.nn.Conv2d(
                in_channels=32,
                out_channels=1,
                kernel_size=(5, 5),
                stride=(1, 1),
                padding="SAME",
                padding_mode="ZEROS",
                use_bias=True,
                key=subkey5,
            ),  # Final regular convolution to 1 x 128 x 128.
            eqx.nn.Lambda(
                lambda x: jnp.tanh(x)
            ),  # Final activation to map to signed unit interval [-1, 1].
        )
        return

    def __call__(
        self, x: jax.Array, state: eqx.nn.State, *, key: jax.Array
    ) -> tuple[jax.Array, eqx.nn.State]:
        """Feed-forward pass through the generator neural network.

        **Arguments**

        - `x`: 1D input vector.

        - `state`: The current state of the generator.

        - `key`: A JAX PRNG key meant for calling stochastic layers.

        **Returns**

        A 3D image (Channel, Y, X indexing) cube as a JAX array and the updated
        state of the generator.
        """
        # print(f"SHAPE OF INPUT X: {x.shape}")
        for layer in self.layers:
            # print(f"CURRENTLY CONSIDERING LAYER OF TYPE '{type(layer)}'")
            if isinstance(layer, eqx.nn.StatefulLayer):
                x, state = layer(x, state)
            else:
                x = layer(x)
            # print(f"OUTPUT SHAPE: {x.shape}")
        return x, state

In [ ]:
class GeneratorDisksNIR(eqx.Module):
    """WGAN generator for monochromatic NIR disk images with upsampling.

    **Attributes**

    - `layers`: Tuple containing the different neural network layers. The only
        requirement is that each layer should just be callable using the output of the
        preceding layer, producing a 3D image (Channel, Y, X indexing) cube from an
        input 1D latent vector of given `size_in`.
    - `size_in`: Size of the latent 'noise' input vector to the first layer.
    """

    layers: tuple[Any, ...]  # Different callable layers.
    size_in: int = eqx.field(static=True)  # Size of input latent vector.

    def __init__(self, key: jax.Array) -> None:
        """**Arguments**

        - `key`: JAX PRNG key to initialise the model.
        """
        key, subkey1, subkey2, subkey3, subkey4, subkey5 = jr.split(key, 6)

        self.size_in = 100
        self.layers = (
            eqx.nn.Linear(
                self.size_in, 256 * 16 * 16, use_bias=True, key=subkey1
            ),  # Fully connected from shape (100,) to (256 * 16 * 16,).
            eqx.nn.Lambda(
                lambda x: jax.nn.leaky_relu(x, negative_slope=0.1)
            ),  # Activation.
            eqx.nn.Lambda(
                lambda x: jnp.reshape(x, shape=(256, 16, 16))
            ),  # Reshape to 256 x 16 x 16.
            eqx.nn.Lambda(
                lambda x: jnp.repeat(
                    jnp.repeat(x, repeats=2, axis=1),
                    2,
                    axis=2,
                )
            ),  # Upsample to 256 x 32 x 32.
            eqx.nn.Conv2d(
                in_channels=256,
                out_channels=128,
                kernel_size=(3, 3),
                stride=(1, 1),
                padding="SAME",
                padding_mode="REFLECT",
                use_bias=True,
                key=subkey2,
            ),  # Regular convolution to 128 x 32 x 32.
            eqx.nn.Lambda(
                lambda x: jax.nn.leaky_relu(x, negative_slope=0.1)
            ),  # Activation.
            eqx.nn.Lambda(
                lambda x: jnp.repeat(
                    jnp.repeat(x, repeats=2, axis=1),
                    2,
                    axis=2,
                )
            ),  # Upsample to 128 x 64 x 64.
            eqx.nn.Conv2d(
                in_channels=128,
                out_channels=64,
                kernel_size=(3, 3),
                stride=(1, 1),
                padding="SAME",
                padding_mode="REFLECT",
                use_bias=True,
                key=subkey3,
            ),  # Regular convolution to 64 x 64 x 64.
            eqx.nn.Lambda(
                lambda x: jax.nn.leaky_relu(x, negative_slope=0.1)
            ),  # Activation.
            eqx.nn.Lambda(
                lambda x: jnp.repeat(
                    jnp.repeat(x, repeats=2, axis=1),
                    repeats=2,
                    axis=2,
                )
            ),  # Upsample to 64 x 128 x 128.
            eqx.nn.Conv2d(
                in_channels=64,
                out_channels=32,
                kernel_size=(3, 3),
                stride=(1, 1),
                padding="SAME",
                padding_mode="REFLECT",
                use_bias=True,
                key=subkey4,
            ),  # Regular convolution to 32 x 128 x 128.
            eqx.nn.Lambda(
                lambda x: jax.nn.leaky_relu(x, negative_slope=0.1)
            ),  # Activation.
            eqx.nn.Conv2d(
                in_channels=32,
                out_channels=1,
                kernel_size=(5, 5),
                stride=(1, 1),
                padding="SAME",
                padding_mode="REFLECT",
                use_bias=True,
                key=subkey5,
            ),  # Final regular convolution to 1 x 128 x 128.
            eqx.nn.Lambda(
                lambda x: jnp.tanh(x)
            ),  # Final activation to map to signed unit interval [-1, 1].
        )
        return

    def __call__(
        self, x: jax.Array, state: eqx.nn.State, *, key: jax.Array
    ) -> tuple[jax.Array, eqx.nn.State]:
        """Feed-forward pass through the generator neural network.

        **Arguments**

        - `x`: 1D input vector.

        - `state`: The current state of the generator.

        - `key`: A JAX PRNG key meant for calling stochastic layers.

        **Returns**

        A 3D image (Channel, Y, X indexing) cube as a JAX array and the updated
        state of the generator.
        """
        # print(f"SHAPE OF INPUT X: {x.shape}")
        for layer in self.layers:
            # print(f"CURRENTLY CONSIDERING LAYER OF TYPE '{type(layer)}'")
            if isinstance(layer, eqx.nn.StatefulLayer):
                x, state = layer(x, state)
            else:
                x = layer(x)
            # print(f"OUTPUT SHAPE: {x.shape}")
        return x, state


class CriticDisksNIR(eqx.Module):
    """WGAN critic for monochromatic NIR disk images. Spectrally normalized to
    maintain 1-Lipschitz continuity.

    **Attributes**

    - `layers`: Tuple containing the different neural network layers. The only
        requirement is that each layer should just be callable using the output of the
        preceding layer. The first layer needs to be able to accept a 3D image (Channel,
        Y, X, indexing) cube.
    """

    layers: tuple[Any, ...]  # Different callable layers.

    def __init__(self, key: jax.Array) -> None:
        """**Arguments**

        - `key`: JAX PRNG key to initialise the model.
        """
        (
            key,
            subkey1,
            subkey2,
            subkey3,
            subkey4,
            subkey5,
            subkey6,
            subkey7,
            subkey8,
        ) = jr.split(key, 9)

        self.layers = (
            eqx.nn.SpectralNorm(
                eqx.nn.Conv2d(
                    in_channels=1,
                    out_channels=32,
                    kernel_size=(5, 5),
                    stride=(1, 1),
                    padding="SAME",
                    padding_mode="ZEROS",
                    use_bias=True,
                    key=subkey1,
                ),
                weight_name="weight",
                num_power_iterations=2,
                key=subkey2,
            ),  # Regular convolution to 32 x 128 x 128 (input is 1 x 128 x 128).
            eqx.nn.SpectralNorm(
                eqx.nn.Conv2d(
                    in_channels=32,
                    out_channels=64,
                    kernel_size=(3, 3),
                    stride=(2, 2),
                    padding="SAME",
                    padding_mode="ZEROS",
                    use_bias=True,
                    key=subkey1,
                ),
                weight_name="weight",
                num_power_iterations=2,
                key=subkey2,
            ),  # Regular convolution to 64 x 64 x 64.
            eqx.nn.Lambda(
                lambda x: jax.nn.leaky_relu(x, negative_slope=0.1)
            ),  # Activation.
            eqx.nn.SpectralNorm(
                eqx.nn.Conv2d(
                    in_channels=64,
                    out_channels=128,
                    kernel_size=(3, 3),
                    stride=(2, 2),
                    padding="SAME",
                    padding_mode="ZEROS",
                    use_bias=True,
                    key=subkey3,
                ),
                weight_name="weight",
                num_power_iterations=2,
                key=subkey4,
            ),  # Regular convolution to 128 x 32 x 32.
            eqx.nn.Lambda(
                lambda x: jax.nn.leaky_relu(x, negative_slope=0.1)
            ),  # Activation.
            eqx.nn.SpectralNorm(
                eqx.nn.Conv2d(
                    in_channels=128,
                    out_channels=256,
                    kernel_size=(3, 3),
                    stride=(2, 2),
                    padding="SAME",
                    padding_mode="ZEROS",
                    use_bias=True,
                    key=subkey5,
                ),
                weight_name="weight",
                num_power_iterations=2,
                key=subkey6,
            ),  # Regular convolution to 256 x 16 x 16.
            eqx.nn.Lambda(
                lambda x: jax.nn.leaky_relu(x, negative_slope=0.1)
            ),  # Activation.
            eqx.nn.Lambda(lambda x: jnp.ravel(x)),  # Ravel to shape (256 * 16 * 16,).
            eqx.nn.SpectralNorm(
                eqx.nn.Linear(256 * 16 * 16, 1, use_bias=True, key=subkey7),
                weight_name="weight",
                num_power_iterations=2,
                key=subkey8,
            ),  # Dense map to scalar of shape (1,) (no further activation needed).
        )

        return

    def __call__(
        self, x: jax.Array, state: eqx.nn.State, *, key: jax.Array
    ) -> tuple[jax.Array, eqx.nn.State]:
        """Feed-forward pass through the neural network.

        **Arguments**

        - `x`: A 3D image (Channel, Y, X indexing) cube array.

        - `state`: The current state of the critic.

        - `key`: A JAX PRNG key meant for calling stochastic layers.

        **Returns**

        A shape (1,) scalar meant to represent the WGAN critic output and the updated
        state of the critic.
        """
        # print("\n")
        # print(f"SHAPE OF INPUT X: {x.shape}")
        for layer in self.layers:
            if isinstance(layer, eqx.nn.StatefulLayer):
                # print(f"CURRENTLY CONSIDERING STATEFUL LAYER OF TYPE '{type(layer)}'")
                x, state = layer(x, state)
            else:
                # print(f"CURRENTLY CONSIDERING NORMAL LAYER OF TYPE '{type(layer)}'")
                x = layer(x)
            # print(f"OUTPUT SHAPE: {x.shape}")
        return x, state

In [ ]:
key = jr.key(42)
key, subkey1, subkey2, subkey3, subkey4, subkey5 = jr.split(key, 6)

gen_notrans, gen_notrans_state = eqx.nn.make_with_state(GeneratorDisksNIR)(key=subkey1)
gen_trans, gen_trans_state = eqx.nn.make_with_state(GeneratorDisksNIROld)(key=subkey2)

print(gen_notrans_state, gen_trans_state)

z_test = jr.normal(key=subkey3, shape=(100,))
test_pass_notrans, gen_notrans_state = gen_notrans(
    z_test, gen_notrans_state, key=subkey4
)
test_pass_trans, gen_trans_state = gen_trans(z_test, gen_trans_state, key=subkey5)

print(gen_notrans_state, gen_trans_state)

fig, ax = plt.subplots(1, 2, figsize=(10, 10))
im = ax[0].imshow(test_pass_notrans[0, :, :])
divider = make_axes_locatable(ax[0])
cax = divider.append_axes("right", size="5%", pad=0.1)
fig.colorbar(im, cax=cax)
ax[0].set_title("Upsampling + Reg. Conv2d")
im = ax[1].imshow(test_pass_trans[0, :, :])
divider = make_axes_locatable(ax[1])
cax = divider.append_axes("right", size="5%", pad=0.1)
fig.colorbar(im, cax=cax)
ax[1].set_title("Transposed Conv2d")
plt.show()

## Fuction definitions

### Do a training loop to see if things work.

In [ ]:
training_img_directory = Path(
    "/home/toond/Documents/phd/paper_iras08_time_series/"
    "training_data/discs_mcfost/runs/ozstar_2zone_cont_training_set/saved_imgs/"
)
training_img_files = sorted(training_img_directory.glob("*.npy"))

training_img_loader = TrainingImgLoader(
    training_img_files,
    batch_size=64,
    normalize=True,
    zoomf=(-0.15, 0.15),
    bkgf=(0.00, 0.10),
    pbkg=0.25,
    blur=1.0,
    rotate=True,
    fliph=True,
    flipv=True,
    read_mode="NUMPY",
    seed=42,
)

# Initialize the generator and critic
key, subkey1, subkey2, subkey3 = jr.split(key, 4)
gen, gen_state = eqx.nn.make_with_state(GeneratorDisksNIR)(key=subkey1)
crit, crit_state = eqx.nn.make_with_state(CriticDisksNIR)(key=subkey2)

# Split generator and critic into desired trainable vs. non-trainable components.
gen_params, gen_static = eqx.partition(gen, filter_spec=eqx.is_array)
crit_params, crit_static = eqx.partition(crit, filter_spec=eqx.is_array)

# Set the control logic for the optimizers.
opt_gen = optax.adam(learning_rate=2e-5, b1=0.0, b2=0.9)
opt_crit = optax.adam(learning_rate=1e-4, b1=0.0, b2=0.9)

# Initialize the generator and critic optimizer states. Note we only want
# to select out the relevant parts to be optimized.
opt_gen_state = opt_gen.init(gen_params)
opt_crit_state = opt_crit.init(crit_params)

(
    gen_params,
    gen_static,
    gen_state,
    crit_params,
    crit_static,
    crit_state,
    opt_gen_state,
    opt_crit_state,
    gen_losses,
    crit_losses,
    scores_training_imgs,
    scores_gen_imgs,
) = train_wgan(
    gen_params,
    gen_static,
    gen_state,
    crit_params,
    crit_static,
    crit_state,
    opt_gen,
    opt_gen_state,
    opt_crit,
    opt_crit_state,
    training_img_loader,
    output_dir="./wgan_train_test_module_form",
    ngen=20,
    ncrit_ratio=5,
    key=subkey3,
    ncheck=10,
)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(6, 9))
axes[0].plot(range(crit_losses.size), -crit_losses)
axes[0].set_xlabel("Number of generator updates")
axes[0].set_ylabel("Critic Wasserstein estimate")

axes[1].plot(range(gen_losses.size), gen_losses)
axes[1].set_xlabel("Number of generator updates")
axes[1].set_ylabel("Generator loss")

axes[2].plot(
    range(scores_training_imgs.size), scores_training_imgs, label="training images"
)
axes[2].plot(
    range(scores_training_imgs.size), scores_gen_imgs, label="generated images"
)
axes[2].set_xlabel("Number of generator updates")
axes[2].set_ylabel("Critic batch score")
axes[2].legend()
fig.tight_layout()
fig.show()